# Flux post-processing chain: IRGA75 input, Level-1 fluxes and meteo (2004-2017, 2019)

**notebook version**: `1` (28 Oct 2026)

## ℹ️ About this notebook

Joins the merged **Level-1** fluxes of the **IRGA75** gas analyser (LI-7500, `20_L1_FLUXES/IRGA75/99`)
with the merged meteo products (`10_METEO/30_PRODUCTS/99`) on one 30MIN index, and stores the result
in the external data folder as parquet. The open-path correction with opco reads this file: its
drivers come from both column sets.

No value is computed here. The flux columns are what `20_L1_FLUXES/IRGA75/99` wrote, without the
self-heating correction (`noSHC`), and the meteo columns exactly what `99` wrote. One correction is
made: the flux records of 6 to 12 July 2013 carry timestamps 7 hours late, from the clock of the
data acquisition, and are moved back to their true time (*Clock shift of July 2013*).
The joined record covers full calendar years, from `PERIOD_START`, 1 January 2004 00:15, to
`PERIOD_END`, 31 December 2019 23:45. The flux columns are empty before the first IRGA75 half-hour on
30 March 2004, from 1 January 2018 to 17 January 2019, where no IRGA75 run exists, and after the last
half-hour on 20 March 2019. The meteo columns hold values throughout.

**This file has to be rebuilt whenever the flux file or one of its meteo columns changes.**

***

## 📇 Data sources

- `20_L1_FLUXES/IRGA75/99_IRGA75_L1_MERGED_noSHC_2004-2017_2019.parquet`: the Level-1 fluxes, one
  record per half-hour, each record taken whole from one EddyPro run.
- `10_METEO/30_PRODUCTS/99_METEO_MERGED_2004-2026.09.parquet`: every merged meteo series with its
  provenance flags. Its own information file, `99_METEO_MERGED_2004-2026.09_INFO.md`, describes
  the columns and flag codes.

Both live in the external data folder and are addressed by absolute path. Both are indexed on
`TIMESTAMP_MIDDLE`, local standard time (UTC+01:00), so they are joined without any shift.

**The meteo columns are those of `METEOCOLS`**, the same tower series as in the IRGA72 input file,
the measured and the homogenised columns side by side. The opco correction uses the homogenised air
temperature and vapour pressure deficit (`OPCO_METEOCOLS`). The meteo names end in `_T1_47_1` and
the EddyPro names never do, which the checks below assert. The EddyPro biomet columns were already
dropped in `20_L1_FLUXES/IRGA75/99`.

## ⚙️ Settings

In [1]:
# --- Sources -----------------------------------------------------------------------------------
# Both inputs live in the external (untracked) data folder and are addressed by absolute path, so
# the notebook does not depend on the working directory.
DATADIR = (r"F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data"
           r"\workflow")
FLUXFILE = rf"{DATADIR}\20_L1_FLUXES\IRGA75\99_IRGA75_L1_MERGED_noSHC_2004-2017_2019.parquet"
METEOFILE = rf"{DATADIR}\10_METEO\30_PRODUCTS\99_METEO_MERGED_2004-2026.09.parquet"

# --- Period ------------------------------------------------------------------------------------
# TIMESTAMP_MIDDLE of the first and the last record: full calendar years around the flux record,
# which runs from 30 March 2004 to 20 March 2019. The meteo record continues to 2026 and is cut.
PERIOD_START = '2004-01-01 00:15:00'
PERIOD_END = '2019-12-31 23:45:00'

# --- Clock shift of July 2013 -------------------------------------------------------------------
# The clock of the data acquisition ran 7 hours ahead from 6 July 2013, about 08:00, until it was
# reset on 12 July 2013 (fieldbook: changed at 16:44 from 22:43:50 to 15:44). The flux records
# labelled CLOCK_SHIFT_FIRST to CLOCK_SHIFT_LAST (TIMESTAMP_MIDDLE) belong CLOCK_SHIFT earlier. Before
# the block the clock jumped over 7 hours, which are empty; the half-hours of 12 July between the
# block and the reset were overwritten in the raw files and stay empty. Docs: Yearly Notes, 2013.
CLOCK_SHIFT_FIRST = '2013-07-06 15:15:00'
CLOCK_SHIFT_LAST = '2013-07-12 19:45:00'
CLOCK_SHIFT = '7h'
CLOCK_SHIFT_CHECK_DAYS = ('2013-07-07', '2013-07-11')   # whole days inside the block

# --- Output ------------------------------------------------------------------------------------
OUTNAME = "00_FPPC_IRGA75_INPUT_noSHC_2004-2017_2019"
OUTPATH = rf"{DATADIR}\30_FLUX_POST-PROCESSING_CHAIN\IRGA75"

# --- Meteo columns -----------------------------------------------------------------------------
# The tower meteo of the products 01 to 07, with their flags, as in the IRGA72 input file. The other
# series of 99 (precipitation, soil, outgoing radiation) are not needed by the chain.
METEOCOLS = [
    'SW_IN_T1_47_1_gfXG', 'FLAG_SW_IN_T1_47_1_ISFILLED',
    'TA_T1_47_1_gfXG', 'FLAG_TA_T1_47_1_ISFILLED', 'TA_T1_47_1_HOMOGENIZED_gfXG', 'FLAG_TA_T1_47_1_SOURCE',
    'PPFD_IN_T1_47_1_gfXG', 'FLAG_PPFD_IN_T1_47_1_ISFILLED',
    'RH_T1_47_1', 'FLAG_RH_T1_47_1_MISSING', 'RH_T1_47_1_HOMOGENIZED', 'FLAG_RH_T1_47_1_SOURCE',
    'PA_T1_47_1',
    'LW_IN_T1_47_1', 'FLAG_LW_IN_T1_47_1_SOURCE',
    'VPD_T1_47_1', 'FLAG_VPD_T1_47_1_ISFILLED', 'FLAG_VPD_T1_47_1_SOURCE', 'VPD_T1_47_1_HOMOGENIZED',
]

# --- Columns the opco correction reads ---------------------------------------------------------
# The inputs of opco.physmodel.drivers(): flux columns from the Level-1 file, meteo drivers from 99.
OPCO_FLUXCOLS = ['FC', 'FC_SSITC_TEST', 'H', 'USTAR', 'WS', 'CO2_MOLAR_DENSITY', 'H2O_MOLAR_DENSITY',
                 'PA_EP', 'AIR_DENSITY', 'AIR_CP', 'CUSTOM_AGC_IRGA75_MEAN']
OPCO_METEOCOLS = ['TA_T1_47_1_HOMOGENIZED_gfXG', 'SW_IN_T1_47_1_gfXG', 'VPD_T1_47_1_HOMOGENIZED']

# --- Checks ------------------------------------------------------------------------------------
REQUIRED_TIME_RESOLUTION = '30min'
METEO_SUFFIX = '_T1_47_1'  # every meteo column name contains it, and no EddyPro column name does
# The drivers of the Level-4.1 gap-filling. They and the opco meteo drivers must be complete over
# the whole joined period.
DRIVERS = ['TA_T1_47_1_gfXG', 'SW_IN_T1_47_1_gfXG', 'VPD_T1_47_1']
FLUXCOLS = ['FC', 'LE', 'H']
# The gap-filling flag of each driver: 0 is measured, any other code a filled value.
FILLFLAGS = {'SW_IN': 'FLAG_SW_IN_T1_47_1_ISFILLED', 'TA': 'FLAG_TA_T1_47_1_ISFILLED',
             'VPD': 'FLAG_VPD_T1_47_1_ISFILLED'}

In [2]:
from datetime import datetime
from pathlib import Path
import importlib.metadata
import textwrap

import numpy as np
import pandas as pd

from diive.core.io.files import save_parquet, load_parquet

pd.set_option('display.max_rows', 60)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)


def tabcap(text, width=100):
    """Print a table caption, above the table it belongs to."""
    print(textwrap.fill(f"Table - {text}", width) + "\n")


# Start time is kept so the last cell can report the total runtime of the notebook.
NOTEBOOK_START = datetime.now()
print(f"Last run: {NOTEBOOK_START.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"diive version: v{importlib.metadata.version('diive')}")

Last run: 2026-10-30 00:43:53
diive version: v0.93.0


## 📂 Load

In [3]:
flux = load_parquet(filepath=FLUXFILE)
meteo = load_parquet(filepath=METEOFILE)
_missing = [c for c in METEOCOLS if c not in meteo.columns]
assert not _missing, f"meteo columns not in {METEOFILE}: {_missing}"
print(f"meteo: {len(METEOCOLS)} of the {meteo.shape[1]} columns of 99 are used")
meteo = meteo[METEOCOLS]

for _name, _df in [('fluxes', flux), ('meteo', meteo)]:
    print(f"{_name:<7} {len(_df):>8,} records x {_df.shape[1]:>3} columns  "
          f"{_df.index[0]} -> {_df.index[-1]}  ({_df.index.name})")
    assert _df.index.name == 'TIMESTAMP_MIDDLE', f"{_name} is not indexed on TIMESTAMP_MIDDLE"
    assert not _df.index.duplicated().any(), f"{_name} has duplicate timestamps"
    _steps = _df.index.to_series().diff().dropna().unique()
    assert len(_steps) == 1 and _steps[0] == pd.Timedelta(REQUIRED_TIME_RESOLUTION), (
        f"{_name} is not on a continuous {REQUIRED_TIME_RESOLUTION} index")

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\20_L1_FLUXES\IRGA
75\99_IRGA75_L1_MERGED_noSHC_2004-2017_2019.parquet (2.004 seconds).

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\10_METEO\30_PRODU
CTS\99_METEO_MERGED_2004-2026.09.parquet (0.395 seconds).

meteo: 19 of the 61 columns of 99 are used
fluxes   262,473 records x 320 columns  2004-03-30 14:45:00 -> 2019-03-20 18:45:00  (TIMESTAMP_MIDDLE)
meteo    398,832 records x  19 columns  2004-01-01 00:15:00 -> 2026-09-30 23:45:00  (TIMESTAMP_MIDDLE)


## 🔗 Join

In [4]:
# The two column sets must not share a name, or one would silently replace the other.
assert all(METEO_SUFFIX in _c for _c in meteo.columns), "a meteo column lacks the meteo suffix"
assert not any(METEO_SUFFIX in _c for _c in flux.columns), "a flux column carries the meteo suffix"
assert not set(flux.columns) & set(meteo.columns), "the two inputs share a column name"

# The joined period runs from PERIOD_START to PERIOD_END on the meteo index, which covers it.
# The flux record must lie inside it; outside its first and last record the flux columns are empty.
_start, _end = pd.Timestamp(PERIOD_START), pd.Timestamp(PERIOD_END)
assert meteo.index[0] <= _start and meteo.index[-1] >= _end, "the meteo record does not cover the period"
assert flux.index[0] >= _start and flux.index[-1] <= _end, "the flux record lies outside the period"
df = meteo.loc[_start:_end].join(flux, how='left')
df = df.reindex(sorted(df.columns), axis=1)

print(f"Joined: {len(df):,} records x {df.shape[1]} columns "
      f"({flux.shape[1]} flux, {meteo.shape[1]} meteo), {df.index[0]} -> {df.index[-1]}")

Joined: 280,512 records x 339 columns (320 flux, 19 meteo), 2004-01-01 00:15:00 -> 2019-12-31 23:45:00


### 📋 Table: the meteo columns over the joined period

In [5]:
_rows = []
for _c in meteo.columns:
    _s = df[_c]
    _rows.append({'column': _c, 'present_%': round(100 * _s.notna().mean(), 2),
                  'first': _s.first_valid_index(), 'last': _s.last_valid_index()})
tabcap("every meteo column of the joined file, the share of the joined period it holds a value for, "
       "and its first and last value. PA and LW_IN keep their outages, because neither is gap-filled")
display(pd.DataFrame(_rows).set_index('column'))

Table - every meteo column of the joined file, the share of the joined period it holds a value for,
and its first and last value. PA and LW_IN keep their outages, because neither is gap-filled



,present_%,first,last
column,,,
SW_IN_T1_47_1_gfXG,100.00,2004-01-01 00:15:00,2019-12-31 23:45:00
FLAG_SW_IN_T1_47_1_ISFILLED,100.00,2004-01-01 00:15:00,2019-12-31 23:45:00
TA_T1_47_1_gfXG,100.00,2004-01-01 00:15:00,2019-12-31 23:45:00
FLAG_TA_T1_47_1_ISFILLED,100.00,2004-01-01 00:15:00,2019-12-31 23:45:00
TA_T1_47_1_HOMOGENIZED_gfXG,100.00,2004-01-01 00:15:00,2019-12-31 23:45:00
FLAG_TA_T1_47_1_SOURCE,100.00,2004-01-01 00:15:00,2019-12-31 23:45:00
PPFD_IN_T1_47_1_gfXG,100.00,2004-01-01 00:15:00,2019-12-31 23:45:00
FLAG_PPFD_IN_T1_47_1_ISFILLED,100.00,2004-01-01 00:15:00,2019-12-31 23:45:00
RH_T1_47_1,100.00,2004-01-01 00:15:00,2019-12-31 23:45:00


### 📋 Table: gap-filled share of the drivers and flux records per year

In [6]:
_year = df.index.year
_tab = pd.DataFrame({f"{_v}_filled_%": (df[_f] > 0).groupby(_year).mean().mul(100).round(1)
                     for _v, _f in FILLFLAGS.items()})
_tab['FC_records'] = df['FC'].notna().groupby(_year).sum()
_tab.index.name = 'year'
tabcap("per calendar year, the share of half-hours whose SW_IN, TA and VPD value is gap-filled "
       "(flag code other than 0), and the number of half-hours with an FC value")
display(_tab)

Table - per calendar year, the share of half-hours whose SW_IN, TA and VPD value is gap-filled (flag
code other than 0), and the number of half-hours with an FC value



,SW_IN_filled_%,TA_filled_%,VPD_filled_%,FC_records
year,,,,
2004,100.0,72.0,72.0,12011
2005,70.3,0.2,0.2,16486
2006,0.3,0.1,0.2,16088
2007,0.2,0.1,0.1,16438
2008,0.3,0.1,0.1,16305
2009,3.0,2.8,2.8,16186
2010,1.0,1.0,1.0,15387
2011,0.2,0.1,0.1,15795
2012,7.3,7.2,7.2,14612


## 🔍 Checks

In [7]:
_period = meteo.loc[pd.Timestamp(PERIOD_START):pd.Timestamp(PERIOD_END)].index
assert df.index.equals(_period), "the joined index is not PERIOD_START to PERIOD_END"
assert df.index[0] == pd.Timestamp(PERIOD_START) and df.index[-1] == pd.Timestamp(PERIOD_END)
assert len(df.columns) == flux.shape[1] + meteo.shape[1], "a column was lost or duplicated"

# Nothing was changed on the way: every column equals its source.
# Before the first and after the last flux record every flux column is empty.
_outside = (df.index < flux.index[0]) | (df.index > flux.index[-1])
assert df.loc[_outside, flux.columns].isna().all().all(), "flux values outside the flux record"
for _c in flux.columns:
    assert df.loc[flux.index, _c].equals(flux[_c]), f"flux column {_c} changed in the join"
for _c in meteo.columns:
    assert df[_c].equals(meteo.loc[_period, _c]), f"meteo column {_c} changed in the join"
print(f"{int(_outside.sum()):,} records outside the flux record ({flux.index[0]} -> {flux.index[-1]}) "
      f"carry meteo only")

# The gap-filling drivers and the opco meteo drivers must hold a value at every timestamp.
_complete = list(dict.fromkeys(DRIVERS + OPCO_METEOCOLS))
for _c in _complete:
    _n = int(df[_c].isna().sum())
    assert _n == 0, f"driver {_c} has {_n} missing values over the joined period"

# Every column the opco correction reads is in the file.
_missing = [c for c in OPCO_FLUXCOLS + OPCO_METEOCOLS if c not in df.columns]
assert not _missing, f"columns for opco not in the joined file: {_missing}"
print(f"Index, columns and values verified. Drivers complete: {', '.join(_complete)}")
print(f"opco columns present: {len(OPCO_FLUXCOLS)} flux, {len(OPCO_METEOCOLS)} meteo")

18,039 records outside the flux record (2004-03-30 14:45:00 -> 2019-03-20 18:45:00) carry meteo only
Index, columns and values verified. Drivers complete: TA_T1_47_1_gfXG, SW_IN_T1_47_1_gfXG, VPD_T1_47_1, TA_T1_47_1_HOMOGENIZED_gfXG, VPD_T1_47_1_HOMOGENIZED
opco columns present: 11 flux, 3 meteo


## ⏱️ Clock shift of July 2013

The clock of the data acquisition ran 7 hours ahead from 6 July 2013, about 08:00, until it was
reset on 12 July 2013. The raw files of this week carry names 7 hours late, and so do the Level-1
fluxes calculated from them: `H` peaks at about 18:00 instead of 13:00, and the afternoon uptake of
`FC` appears at night. The flux records of the block are moved 7 hours back, into the 7 empty hours
the clock jumped over, and the time columns of the Level-1 file move with them. The meteo columns
come from another logger and are not touched. The check compares the timing of `H` with the
incoming shortwave radiation before and after the move.

In [8]:
_first, _last = pd.Timestamp(CLOCK_SHIFT_FIRST), pd.Timestamp(CLOCK_SHIFT_LAST)
_dt = pd.Timedelta(CLOCK_SHIFT)
_fluxcols = list(flux.columns)


def h_lag_hours(frame, days=CLOCK_SHIFT_CHECK_DAYS):
    """Lag (hours) of H behind the incoming shortwave radiation over whole days, from the shift of
    the radiation that correlates best with H, within -12 to +12 hours."""
    _h = frame.loc[days[0]:f"{days[1]} 23:45", 'H']
    _sw = frame['SW_IN_T1_47_1_gfXG']
    _best = max(range(-24, 25), key=lambda k: _h.corr(_sw.shift(k).reindex(_h.index)))
    return _best / 2


_block = df.loc[_first:_last, _fluxcols].copy()
_target = _block.index - _dt
# The clock jumped over the 7 hours before the block, so they hold no flux record; the block lands
# there and on its own first part.
assert df.loc[_target[0]:_first - pd.Timedelta('30min'), _fluxcols].isna().all().all(), (
    "the hours the clock jumped over hold flux records, so the shift would overwrite measured data")
LAG_BEFORE = h_lag_hours(df)

df.loc[_first:_last, _fluxcols] = np.nan
df.loc[_target, _fluxcols] = _block.to_numpy()
# The time columns of the Level-1 file move with their records. EddyPro's night flag was set from
# the wrong clock and is cleared; the chain tells day from night by potential radiation.
for _c in ('DOY_START', 'DOY_END'):
    df.loc[_target, _c] = _block[_c].to_numpy() - _dt / pd.Timedelta('1D')
df.loc[_target, 'TIMESTAMP_START'] = (_target - pd.Timedelta('15min')).strftime('%Y%m%d%H%M').astype(float)
df.loc[_target, 'NIGHT'] = np.nan
LAG_AFTER = h_lag_hours(df)

_moved = int(_block['FC'].notna().sum())
print(f"moved {len(_block):,} records ({_moved:,} with FC) from {_first} - {_last} "
      f"to {_target[0]} - {_target[-1]}")
print(f"lag of H behind SW_IN, {CLOCK_SHIFT_CHECK_DAYS[0]} to {CLOCK_SHIFT_CHECK_DAYS[1]}: "
      f"{LAG_BEFORE:+.1f} h before, {LAG_AFTER:+.1f} h after")
assert LAG_BEFORE == _dt / pd.Timedelta('1h'), "the block no longer lags by CLOCK_SHIFT before the move"
assert abs(LAG_AFTER) <= 0.5, "H still lags the radiation after the move"
assert df.loc[_target[-1] + pd.Timedelta('30min'):_last, _fluxcols].isna().all().all(), (
    "the half-hours between the moved block and the reset are not empty")

moved 298 records (297 with FC) from 2013-07-06 15:15:00 - 2013-07-12 19:45:00 to 2013-07-06 08:15:00 - 2013-07-12 12:45:00
lag of H behind SW_IN, 2013-07-07 to 2013-07-11: +7.0 h before, +0.0 h after


### 📋 Table: daily lag of H after the move

In [9]:
# Per day, the lag of H behind the radiation after the move, over the week of the shift and a day
# either side.
_rows = {}
for _day in pd.date_range('2013-07-05', '2013-07-13', freq='D'):
    _d = f"{_day:%Y-%m-%d}"
    _n = int(df.loc[_d, 'H'].notna().sum())
    _rows[_d] = {'H records': _n, 'lag of H (h)': h_lag_hours(df, (_d, _d)) if _n >= 24 else np.nan}
tabcap("daily lag of H behind the incoming shortwave radiation after the move, 5 to 13 July 2013. "
       "12 July holds fewer records: its half-hours between the block and the reset of the clock are empty")
display(pd.DataFrame.from_dict(_rows, orient='index'))

Table - daily lag of H behind the incoming shortwave radiation after the move, 5 to 13 July 2013. 12
July holds fewer records: its half-hours between the block and the reset of the clock are empty



,H records,lag of H (h)
2013-07-05,48,0.0
2013-07-06,48,0.0
2013-07-07,48,0.0
2013-07-08,48,0.0
2013-07-09,48,0.0
2013-07-10,48,0.0
2013-07-11,48,0.0
2013-07-12,34,0.5
2013-07-13,48,0.5


## 💾 Export

In [10]:
Path(OUTPATH).mkdir(parents=True, exist_ok=True)
filepath = save_parquet(filename=OUTNAME, data=df, outpath=OUTPATH)
print(f"Saved to: {filepath}")

> Saved file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\30_FLUX_POST-PROC
ESSING_CHAIN\IRGA75\00_FPPC_IRGA75_INPUT_noSHC_2004-2017_2019.parquet (3.593 seconds).

Saved to: F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\30_FLUX_POST-PROCESSING_CHAIN\IRGA75\00_FPPC_IRGA75_INPUT_noSHC_2004-2017_2019.parquet


## ✅ Read the written file back

In [11]:
_written = load_parquet(filepath=str(Path(OUTPATH) / f"{OUTNAME}.parquet"))

print(f"index:   {_written.index[0]} -> {_written.index[-1]}  ({len(_written):,} records)")
print(f"columns: {len(_written.columns):,}")

assert _written.index.equals(df.index), "index changed on write"
assert list(_written.columns) == list(df.columns), "columns changed on write"
for _c in dict.fromkeys(FLUXCOLS + DRIVERS + OPCO_FLUXCOLS + OPCO_METEOCOLS):
    assert _written[_c].equals(df[_c]), f"{_c} changed on write"
print("\nWritten file verified.")

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\30_FLUX_POST-PROC
ESSING_CHAIN\IRGA75\00_FPPC_IRGA75_INPUT_noSHC_2004-2017_2019.parquet (0.363 seconds).

index:   2004-01-01 00:15:00 -> 2019-12-31 23:45:00  (280,512 records)
columns: 339

Written file verified.


***

# End of notebook.

In [12]:
NOTEBOOK_END = datetime.now()
_h, _rem = divmod(int((NOTEBOOK_END - NOTEBOOK_START).total_seconds()), 3600)
_m, _s = divmod(_rem, 60)

print(f"Started:  {NOTEBOOK_START.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Finished: {NOTEBOOK_END.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Runtime:  {_h:02d}:{_m:02d}:{_s:02d} (hh:mm:ss)")

Started:  2026-10-30 00:43:53
Finished: 2026-10-30 00:44:08
Runtime:  00:00:14 (hh:mm:ss)
